# Translation en -> fr with a manual training loop (Accelerate)

Same task as `translation_classic_nlp_task.ipynb` (Marian `opus-mt-en-fr` fine-tuned on KDE4), but trained with a hand-written loop and `accelerate` instead of `Seq2SeqTrainer`. Hyperparameters match the Trainer run so the results can be compared.

In [1]:
import time
import json
import numpy as np
import torch
from accelerate.utils import set_seed
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

set_seed(42)  # Trainer default seed

checkpoint = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

print(torch.cuda.is_available())

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 45732.84it/s]


True


In [2]:
raw_datasets = load_dataset('Helsinki-NLP/kde4', lang1='en', lang2='fr', trust_remote_code=True)

split_datasets = raw_datasets['train'].train_test_split(train_size=0.9, seed=42)
split_datasets['validation'] = split_datasets.pop('test')
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 189155
    })
    validation: Dataset({
        features: ['id', 'translation'],
        num_rows: 21018
    })
})

In [3]:
max_length = 128

def preprocess(example):
    inputs = [ex['en'] for ex in example['translation']]
    target = [ex['fr'] for ex in example['translation']]
    return tokenizer(inputs, text_target=target, max_length=max_length, truncation=True)

tokenized_datasets = split_datasets.map(
    preprocess, batched=True, remove_columns=split_datasets['train'].column_names
)

In [4]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# sort validation by length so each generation batch has similar-length sentences (less padding, same results)
eval_dataset = (
    tokenized_datasets['validation']
    .map(lambda x: {'len': len(x['input_ids'])})
    .sort('len', reverse=True)
    .remove_columns('len')
)

train_loader = DataLoader(
    tokenized_datasets['train'],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=32,
)
eval_loader = DataLoader(
    eval_dataset,
    collate_fn=data_collator,
    batch_size=128,
)

In [5]:
from torch.optim import AdamW

# same as Trainer: weight decay on everything except biases and LayerNorm weights
no_decay = ['bias', 'layer_norm.weight']
param_groups = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer = AdamW(param_groups, lr=2e-5)

In [6]:
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision='fp16')
model, optimizer, train_loader, eval_loader = accelerator.prepare(
    model, optimizer, train_loader, eval_loader
)
accelerator.device

device(type='cuda')

In [7]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
num_training_steps

17736

In [8]:
import evaluate

metric = evaluate.load('sacrebleu')

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

In [9]:
def evaluate_model():
    model.eval()
    losses = []
    for batch in eval_loader:
        with torch.no_grad():
            losses.append(model(**batch).loss.item())
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch['input_ids'],
                attention_mask=batch['attention_mask'],
                max_length=max_length,
            )
        labels = batch['labels']

        # pad before gathering so tensors from different processes have the same length
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizer.pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        decoded_preds, decoded_labels = postprocess(
            accelerator.gather(generated_tokens), accelerator.gather(labels)
        )
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    return float(np.mean(losses)), metric.compute()['score']

In [ ]:
history = []
log_every = 1000
step = 0
start = time.time()

for epoch in range(num_epochs):
    model.train()
    epoch_start = time.time()
    running_loss = 0.0
    for batch in train_loader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)
        accelerator.clip_grad_norm_(model.parameters(), 1.0)  # Trainer default max_grad_norm

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        step += 1
        if step % log_every == 0:
            print(f'step {step}/{num_training_steps}  loss {loss.item():.4f}  elapsed {(time.time() - start) / 60:.1f} min')

    train_loss = running_loss / len(train_loader)
    train_minutes = (time.time() - epoch_start) / 60

    eval_start = time.time()
    eval_loss, bleu = evaluate_model()
    eval_minutes = (time.time() - eval_start) / 60

    history.append({
        'epoch': epoch + 1,
        'train_loss': round(train_loss, 4),
        'eval_loss': round(eval_loss, 4),
        'bleu': round(bleu, 2),
        'train_minutes': round(train_minutes, 1),
        'eval_minutes': round(eval_minutes, 1),
    })
    print(f'epoch {epoch + 1}: train_loss {train_loss:.4f}  eval_loss {eval_loss:.4f}  BLEU {bleu:.2f}')

total_minutes = (time.time() - start) / 60
print(f'total: {total_minutes:.1f} min')

In [ ]:
import pandas as pd

pd.DataFrame(history)

In [ ]:
# a few translations from the fine-tuned model
samples = [
    'Default to expanded threads',
    'Show the toolbar',
    'Unable to import %1 using the OFX importer plugin. This file is not the correct format.',
    'Save the current document',
    'Open a new window',
]

model.eval()
unwrapped = accelerator.unwrap_model(model)
inputs = tokenizer(samples, return_tensors='pt', padding=True).to(accelerator.device)
with torch.no_grad():
    out = unwrapped.generate(**inputs, max_length=max_length)
translations = tokenizer.batch_decode(out, skip_special_tokens=True)

for en, fr in zip(samples, translations):
    print(f'{en}  ->  {fr}')

In [ ]:
# results go to a file; model weights are not kept (only the pipeline goes to GitHub)
results = {
    'method': 'manual loop + accelerate (fp16)',
    'model': checkpoint,
    'hyperparameters': {
        'learning_rate': 2e-5,
        'weight_decay': 0.01,
        'train_batch_size': 32,
        'eval_batch_size': 128,
        'epochs': num_epochs,
        'max_length': max_length,
        'scheduler': 'linear, no warmup',
    },
    'epochs': history,
    'total_minutes': round(total_minutes, 1),
    'sample_translations': [{'en': en, 'fr': fr} for en, fr in zip(samples, translations)],
}

with open('results/translation_accelerate_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)